# Deepfake Detection using PyTorch

This notebook provides a comprehensive framework for deepfake detection using various pre-trained convolutional neural networks (CNNs) such as ResNet50, EfficientNetB0, DenseNet121, and MobileNetV3. It includes data loading, augmentation, model building, training with early stopping, and detailed evaluation with visualizations.

**Table of Contents:**
1.  [Setup and Imports](#setup-and-imports)
2.  [Deepfake Dataset Class](#deepfake-dataset-class)
3.  [Data Loading and Preprocessing](#data-loading-and-preprocessing)
4.  [Model Architectures](#model-architectures)
    * [ResNet50](#resnet50)
    * [EfficientNetB0](#efficientnetb0)
    * [DenseNet121](#densenet121)
    * [MobileNetV3](#mobilenetv3)
5.  [Training Loop](#training-loop)
6.  [Evaluation and Visualization](#evaluation-and-visualization)
7.  [Configuration and Main Execution](#configuration-and-main-execution)

## 1. Setup and Imports
This section imports all necessary libraries for data handling, model building, training, and evaluation.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from datasets import load_dataset # For Hugging Face datasets
from PIL import Image
import numpy as np
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from datetime import datetime
from torchsummary import summary
from collections import defaultdict, OrderedDict
import warnings
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from typing import Optional, List, Dict, Tuple # Added for type hinting

# Suppress unnecessary warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Set device for training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

Using device: cuda


## 2. Deepfake Dataset Class
This custom PyTorch `Dataset` class handles loading images and labels from a Hugging Face dataset. It includes options for preloading data into memory for faster access and converting grayscale images to RGB.

In [2]:
class DeepfakeDataset(Dataset):
    """Custom PyTorch Dataset for Deepfake classification"""
    
    def __init__(self, hf_dataset, transform=None, preload=False):
        """
        Args:
            hf_dataset: Hugging Face dataset object
            transform: Optional torchvision transforms
            preload: Whether to load all images into memory upfront
        """
        self.dataset = hf_dataset
        self.transform = transform
        self.preload = preload
        
        if preload:
            self.images = []
            self.labels = []
            print("Preloading dataset into memory...")
            for item in tqdm(hf_dataset, desc="Preloading"):
                self.images.append(item['image'])
                self.labels.append(item['label'])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if self.preload:
            image = self.images[idx]
            label = self.labels[idx]
        else:
            item = self.dataset[idx]
            image = item['image']
            label = item['label']
        
        # Convert grayscale to RGB if needed
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)  # Using long for classification

## 3. Data Loading and Preprocessing
This section defines functions for getting image transformations (augmentation for training, simple resizing for validation/testing) and preparing `DataLoader` objects. It also includes logic for handling class imbalance using `WeightedRandomSampler`.

In [9]:
def get_transforms(img_size=224, augment=True, color_jitter=0.2, random_erase_prob=0.1):
    """Return train and validation transforms"""
    if augment:
        train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(
                brightness=color_jitter,
                contrast=color_jitter,
                saturation=color_jitter,
                hue=0.1
            ),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=random_erase_prob, scale=(0.02, 0.2))
        ])
    else:
        train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])
    
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_transform

def prepare_dataloaders(cfg):
    """Prepare train, validation, and test dataloaders from pre-split directories"""
    train_transform, val_transform = get_transforms(
        img_size=cfg.img_size,
        augment=cfg.augment,
        color_jitter=cfg.color_jitter,
        random_erase_prob=cfg.random_erase_prob
    )
    
    # Define paths for pre-split datasets
    train_dir = os.path.join(cfg.data_dir, 'train')
    val_dir = os.path.join(cfg.data_dir, 'val')
    test_dir = os.path.join(cfg.data_dir, 'test')
    
    # Load datasets using ImageFolder and apply respective transforms
    train_dataset = ImageFolder(root=train_dir, transform=train_transform)
    val_dataset = ImageFolder(root=val_dir, transform=val_transform)
    test_dataset = ImageFolder(root=test_dir, transform=val_transform)
    
    # Update class names in config based on the loaded dataset
    cfg.class_names = train_dataset.classes
    with open(os.path.join(cfg.exp_dir, "class_names.json"), 'w') as f:
        json.dump(cfg.class_names, f)
    
    # Handle class imbalance for the training set
    sampler = None
    shuffle = True
    if cfg.class_weights:
        # Get targets from the training dataset
        targets = [label for _, label in train_dataset.samples]
        class_counts = np.bincount(targets)
        class_weights_array = 1. / class_counts
        class_weights_array = class_weights_array / class_weights_array.sum()
        
        sample_weights = [class_weights_array[label] for label in targets]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
        shuffle = False # Sampler handles shuffling
    
    # Create optimized dataloaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=cfg.batch_size, 
        shuffle=shuffle, 
        sampler=sampler,
        num_workers=4, 
        pin_memory=True,
        # persistent_workers=True # Enable if num_workers > 0 and you have enough memory
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=cfg.batch_size * 2, 
        num_workers=4, 
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.batch_size * 2,
        num_workers=4,
        pin_memory=True
    )
    
    return train_loader, val_loader, test_loader

# Example usage (for testing data loading setup)
if __name__ == '__main__':
    # Create a dummy config for testing
    class DummyConfig:
        def __init__(self):
            self.data_dir = "dataset" # Create this directory with 'train/real', 'train/fake', etc.
            self.batch_size = 32
            self.img_size = 224
            self.val_split = 0.1 # These splits are no longer used for ImageFolder, but kept for consistency
            self.test_split = 0.1 # These splits are no longer used for ImageFolder, but kept for consistency
            self.class_weights = True
            self.augment = True
            self.color_jitter = 0.2
            self.random_erase_prob = 0.1
            self.exp_dir = "./dummy_experiments"
            self.class_names = ["Real", "Fake"]
            os.makedirs(self.exp_dir, exist_ok=True)
            
    # Create dummy data for ImageFolder to work
    os.makedirs("./dummy_dataset/train/real", exist_ok=True)
    os.makedirs("./dummy_dataset/train/fake", exist_ok=True)
    os.makedirs("./dummy_dataset/val/real", exist_ok=True)
    os.makedirs("./dummy_dataset/val/fake", exist_ok=True)
    os.makedirs("./dummy_dataset/test/real", exist_ok=True)
    os.makedirs("./dummy_dataset/test/fake", exist_ok=True)

    Image.new('RGB', (224, 224), color = 'red').save('./dummy_dataset/train/real/img1.png')
    Image.new('RGB', (224, 224), color = 'blue').save('./dummy_dataset/train/fake/img2.png')
    Image.new('RGB', (224, 224), color = 'green').save('./dummy_dataset/val/real/img3.png')
    Image.new('RGB', (224, 224), color = 'yellow').save('./dummy_dataset/val/fake/img4.png')
    Image.new('RGB', (224, 224), color = 'purple').save('./dummy_dataset/test/real/img5.png')
    Image.new('RGB', (224, 224), color = 'orange').save('./dummy_dataset/test/fake/img6.png')
    
    dummy_cfg = DummyConfig()
    print("\n📂 Loading and preparing data (dummy test)...")
    try:
        train_loader, val_loader, test_loader = prepare_dataloaders(dummy_cfg)
        sample_imgs, sample_labels = next(iter(train_loader))
        print(f"\n✅ Sample batch shape: {sample_imgs.shape}")
        print(f"✅ Sample labels: {sample_labels[:8]}")
        print(f"✅ Training batches: {len(train_loader)}, Validation batches: {len(val_loader)}, Test batches: {len(test_loader)}")
    except Exception as e:
        print(f"Error during dummy data loading: {e}")
    finally:
        # Clean up dummy data
        import shutil
        shutil.rmtree('./dummy_dataset')
        shutil.rmtree('./dummy_experiments')



📂 Loading and preparing data (dummy test)...

✅ Sample batch shape: torch.Size([2, 3, 224, 224])
✅ Sample labels: tensor([0, 0])
✅ Training batches: 1, Validation batches: 1, Test batches: 1


## 4. Model Architectures
This section defines the functions to build different pre-trained CNN models (ResNet50, EfficientNetB0, DenseNet121, MobileNetV3) with custom classification heads for binary deepfake detection. Each model can be configured to freeze backbone layers, adjust dropout, and use pre-trained weights.

### ResNet50

In [18]:
def build_resnet50(freeze=True, dropout=0.5, pretrained=True):
    """Build ResNet50 model with customizable head"""
    weights = models.ResNet50_Weights.DEFAULT if pretrained else None
    model = models.resnet50(weights=weights)
    
    # Freeze layers if specified
    if freeze:
        for param in model.parameters():
            param.requires_grad = False
    
    # Always unfreeze last layer and optionally layer4
    for name, param in model.named_parameters():
        if name.startswith("layer4") or name.startswith("fc"):
            param.requires_grad = True
    
    # Replace classifier head
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(512, 1),
        nn.Sigmoid() # Sigmoid for binary classification
    )
    
    return model.to(device)

### EfficientNetB0

In [19]:
class EfficientNetB0Binary(nn.Module):
    """Enhanced EfficientNetB0 for binary classification with comprehensive monitoring"""
    
    def __init__(self,
                 freeze_backbone: bool = True,
                 dropout_rate: float = 0.4,
                 pretrained: bool = True,
                 unfreeze_layers: Optional[List[str]] = None,
                 custom_head: Optional[nn.Module] = None,
                 feature_dim: int = 256,
                 gradient_checkpointing: bool = False):
        """
        Initialize EfficientNetB0 model with enhanced capabilities.
        
        Args:
            freeze_backbone: Freeze feature extractor if True
            dropout_rate: Dropout probability (0.0-1.0)
            pretrained: Use ImageNet pretrained weights
            unfreeze_layers: List of layer patterns to unfreeze
            custom_head: Custom classifier head module
            feature_dim: Dimension of intermediate features
            gradient_checkpointing: Enable memory-efficient training
        """
        super().__init__()
        
        # Validate inputs
        if not 0 <= dropout_rate <= 1:
            raise ValueError("dropout_rate must be between 0 and 1")
            
        if feature_dim <= 0:
            raise ValueError("feature_dim must be positive")
        
        # Load pretrained weights with validation
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.base_model = models.efficientnet_b0(weights=weights)
        
        # Enable gradient checkpointing if requested
        if gradient_checkpointing:
            self.base_model.features.gradient_checkpointing = True
            warnings.warn("Gradient checkpointing enabled - tradeoff between memory and speed")
        
        # Configuration tracking
        self.config = {
            'freeze_backbone': freeze_backbone,
            'dropout_rate': dropout_rate,
            'pretrained': pretrained,
            'unfreeze_layers': unfreeze_layers,
            'feature_dim': feature_dim,
            'gradient_checkpointing': gradient_checkpointing,
            'input_size': (3, 224, 224)  # Standard EfficientNetB0 input
        }
        
        # Setup layers
        self._configure_layers(freeze_backbone, unfreeze_layers)
        in_features = self.base_model.classifier[1].in_features
        self.base_model.classifier = self._build_head(
            in_features, feature_dim, dropout_rate, custom_head)
        
        # Device setup
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(self.device)
        
        # Activation monitoring
        self.activations = OrderedDict()
        self.gradients = OrderedDict()
        self._register_hooks()

    def _configure_layers(self, freeze_backbone: bool, unfreeze_layers: Optional[List[str]]):
        """Configure layer freezing with validation"""
        if freeze_backbone:
            for param in self.base_model.parameters():
                param.requires_grad = False
        
        # Default layers to unfreeze (last block + classifier)
        if unfreeze_layers is None:
            unfreeze_layers = ["features.6", "classifier"]
        
        # Validate unfreeze patterns
        valid_layers = set(name for name, _ in self.base_model.named_parameters())
        for pattern in unfreeze_layers:
            if not any(pattern in name for name in valid_layers):
                warnings.warn(f"Unfreeze pattern '{pattern}' doesn't match any layers")
        
        # Apply unfreezing
        for name, param in self.base_model.named_parameters():
            if any(pattern in name for pattern in unfreeze_layers):
                param.requires_grad = True

    def _build_head(self, 
                   in_features: int,
                   feature_dim: int,
                   dropout_rate: float,
                   custom_head: Optional[nn.Module]) -> nn.Module:
        """Build classifier head with feature extraction capability"""
        if custom_head is not None:
            if not isinstance(custom_head, nn.Module):
                raise TypeError("custom_head must be a nn.Module")
            return custom_head
        
        return nn.Sequential(
            OrderedDict([
                ('dropout1', nn.Dropout(p=dropout_rate)),
                ('linear1', nn.Linear(in_features, feature_dim)),
                ('bn1', nn.BatchNorm1d(feature_dim)),
                ('silu', nn.SiLU(inplace=True)),
                ('dropout2', nn.Dropout(p=dropout_rate/2)),
                ('linear2', nn.Linear(feature_dim, 1)),
                ('sigmoid', nn.Sigmoid()) # Sigmoid for binary classification
            ])
        )

    def _register_hooks(self):
        """Register forward and backward hooks to monitor layer activations and gradients"""
        def get_activation_hook(name):
            def hook(module, input, output):
                self.activations[name] = output.detach()
            return hook
        
        def get_gradient_hook(name):
            def hook(module, grad_input, grad_output):
                self.gradients[name] = grad_output[0].detach()
            return hook
        
        # Monitor key layers
        self.handles = []
        for name, layer in self.base_model.named_children():
            if name in ['features.6', 'classifier']: # Example layers to monitor
                # Forward hook
                self.handles.append(
                    layer.register_forward_hook(get_activation_hook(name))
                )
                # Backward hook
                self.handles.append(
                    layer.register_full_backward_hook(get_gradient_hook(name)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with activation tracking"""
        if x.shape[1:] != torch.Size(self.config['input_size']):
            warnings.warn(f"Input size {x.shape[1:]} doesn't match expected {self.config['input_size']}")
        
        self.activations.clear()
        self.gradients.clear()
        return self.base_model(x)

    def get_feature_extractor(self) -> nn.Module:
        """Return feature extractor with avgpool"""
        return nn.Sequential(
            self.base_model.features,
            self.base_model.avgpool
        )

    def get_features(self) -> Dict[str, torch.Tensor]:
        """Get intermediate features from last forward pass"""
        return self.activations

    def get_gradients(self) -> Dict[str, torch.Tensor]:
        """Get gradients from last backward pass"""
        return self.gradients

    def get_trainable_params(self) -> List[str]:
        """Get names of trainable parameters"""
        return [name for name, param in self.named_parameters() 
                if param.requires_grad]

    def get_param_counts(self) -> Tuple[int, int]:
        """Get counts of trainable and total parameters"""
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return trainable, total

    def get_config(self) -> Dict:
        """Get model configuration"""
        return self.config

    def __del__(self):
        """Cleanup hooks when model is deleted"""
        for handle in self.handles:
            handle.remove()

def build_efficientnetb0(
    freeze: bool = True,
    dropout: float = 0.4,
    pretrained: bool = True,
    unfreeze_layers: Optional[List[str]] = None,
    custom_head: Optional[nn.Module] = None,
    feature_dim: int = 256,
    gradient_checkpointing: bool = False
) -> EfficientNetB0Binary:
    """
    Build configured EfficientNetB0Binary with enhanced options.
    
    Args:
        freeze: Freeze base layers
        dropout: Dropout rate (0.0-1.0)
        pretrained: Use pretrained weights
        unfreeze_layers: Layer patterns to unfreeze
        custom_head: Custom classifier
        feature_dim: Intermediate feature dimension
        gradient_checkpointing: Enable memory-efficient training
        
    Returns:
        Configured EfficientNetB0Binary
    """
    return EfficientNetB0Binary(
        freeze_backbone=freeze,
        dropout_rate=dropout,
        pretrained=pretrained,
        unfreeze_layers=unfreeze_layers,
        custom_head=custom_head,
        feature_dim=feature_dim,
        gradient_checkpointing=gradient_checkpointing
    )

### DenseNet121

In [20]:
class DenseNet121Binary(nn.Module):
    """DenseNet121 model for binary classification"""
    def __init__(self, freeze_backbone=True, dropout=0.4, pretrained=True):
        super().__init__()
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        self.base_model = models.densenet121(weights=weights)
        
        if freeze_backbone:
            for param in self.base_model.parameters():
                param.requires_grad = False
        
        # Unfreeze the classifier and the last dense block
        for name, param in self.base_model.named_parameters():
            if "classifier" in name or "denseblock4" in name:
                param.requires_grad = True
                
        num_ftrs = self.base_model.classifier.in_features
        self.base_model.classifier = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 1),
            nn.Sigmoid() # Sigmoid for binary classification
        )
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(self.device)
        
    def forward(self, x):
        return self.base_model(x)

def build_densenet121(freeze=True, dropout=0.4, pretrained=True):
    """Build configured DenseNet121Binary model"""
    return DenseNet121Binary(freeze_backbone=freeze, dropout=dropout, pretrained=pretrained)

### MobileNetV3

In [26]:
class MobileNetV3Binary(nn.Module):
    """MobileNetV3 model for binary classification with a custom head."""
    def __init__(self, freeze_backbone=True, dropout=0.4, pretrained=True):
        super().__init__()
        # Use the correct weights enum for MobileNetV3_Large
        weights = models.MobileNetV3_Large_Weights.DEFAULT if pretrained else None
        self.base_model = models.mobilenet_v3_large(weights=weights)

        if freeze_backbone:
            # Freeze all parameters in the feature extractor
            for param in self.base_model.features.parameters():
                param.requires_grad = False

        # Unfreeze the last block of features and the classifier
        # This loop ensures that even if the backbone was frozen, these specific parts are unfrozen
        for name, param in self.base_model.named_parameters():
            # 'features.16' is typically the last block in MobileNetV3_Large
            # 'classifier' refers to the entire classifier module
            if "features.16" in name or "classifier" in name:
                param.requires_grad = True
                
        # Replace the classifier head with a custom binary classification head
        # MobileNetV3's classifier is often a Sequential with multiple layers.
        # The first layer in the original classifier usually has `in_features`.
        in_features = self.base_model.classifier[0].in_features
        self.base_model.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.Hardswish(inplace=True), # MobileNetV3 uses Hardswish activation
            nn.Dropout(dropout),
            nn.Linear(512, 1), # Output for binary classification (1 neuron)
            nn.Sigmoid() # Sigmoid activation to output probabilities between 0 and 1
        )
        
        # Ensure the model is moved to the correct device (GPU/CPU)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(self.device)

    def forward(self, x):
        """Forward pass through the MobileNetV3 model."""
        return self.base_model(x)

def build_mobilenetv3(freeze=True, dropout=0.4, pretrained=True):
    """Build and configure the MobileNetV3Binary model.
    
    Args:
        freeze (bool): Whether to freeze the backbone layers.
        dropout (float): Dropout rate for the custom classifier head.
        pretrained (bool): Whether to use ImageNet pretrained weights.
        
    Returns:
        MobileNetV3Binary: The configured MobileNetV3 model.
    """
    return MobileNetV3Binary(freeze_backbone=freeze, dropout=dropout, pretrained=pretrained)

## 5. Training Loop
This section implements the core training logic, including forward and backward passes, optimizer steps, learning rate scheduling, mixed precision training, early stopping, and model checkpointing. It also logs metrics to TensorBoard for visualization.

In [27]:
def train_model(model, train_loader, val_loader, cfg):
    """Training loop with early stopping, learning rate scheduling, and mixed precision"""
    # Setup device
    device = next(model.parameters()).device
    
    # Setup optimization
    if cfg.optimizer == "AdamW":
        optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    else:
        optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    
    # Setup learning rate scheduler
    if cfg.scheduler == "ReduceLROnPlateau":
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=cfg.patience // 2, min_lr=cfg.min_lr
        )
    else: # CosineAnnealingWarmRestarts
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=cfg.epochs // 5, T_mult=1, eta_min=cfg.min_lr
        )
    
    # Loss function (BCEWithLogitsLoss combines sigmoid and BCELoss for numerical stability)
    criterion = nn.BCEWithLogitsLoss()
    
    # Mixed precision training scaler
    scaler = GradScaler(enabled=cfg.mixed_precision)
    
    # Initialize tracking
    history = defaultdict(list)
    best_metrics = {'auc': 0.0, 'epoch': 0, 'val_loss': float('inf')}
    early_stop_counter = 0
    writer = SummaryWriter(cfg.log_dir)
    
    # Training loop
    for epoch in range(cfg.epochs):
        # Unfreeze backbone after certain epochs
        if cfg.freeze_backbone and epoch == cfg.freeze_epochs:
            print("\nUnfreezing backbone layers...")
            for param in model.parameters():
                param.requires_grad = True
            # Re-initialize optimizer with all parameters now trainable
            if cfg.optimizer == "AdamW":
                optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
            else:
                optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
            # Re-initialize scheduler with new optimizer
            if cfg.scheduler == "ReduceLROnPlateau":
                scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='max', factor=0.5, patience=cfg.patience // 2, min_lr=cfg.min_lr
                )
            else:
                scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
                    optimizer, T_0=cfg.epochs // 5, T_mult=1, eta_min=cfg.min_lr
                )
        
        # Training phase
        model.train()
        running_loss = 0.0
        train_metrics = defaultdict(float)
        
        for batch_idx, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs}")):
            x, y = x.to(device), y.float().unsqueeze(1).to(device)
            
            optimizer.zero_grad(set_to_none=True)
            
            # Mixed precision forward pass
            with autocast(enabled=cfg.mixed_precision):
                preds = model(x)
                loss = criterion(preds, y)
            
            # Backward pass and optimizer step with scaler
            scaler.scale(loss).backward()
            
            # Gradient clipping (optional, but good practice with mixed precision)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            # Update metrics
            running_loss += loss.item() * x.size(0)
            y_pred = (preds > 0).int() # Convert logits to binary predictions
            train_metrics['acc'] += accuracy_score(y.cpu().numpy(), y_pred.cpu().numpy()) * x.size(0)
        
        # Calculate epoch metrics
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = train_metrics['acc'] / len(train_loader.dataset)
        
        # Validation phase
        val_metrics, _, _, _ = evaluate_model(model, val_loader, criterion, device)
        
        # Learning rate scheduling
        if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_metrics['auc'])
        else:
            scheduler.step()
        
        # Update history
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_auc'].append(val_metrics['auc'])
        history['val_f1'].append(val_metrics['f1'])
        history['lr'].append(optimizer.param_groups[0]['lr'])
        
        # TensorBoard logging
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_metrics['loss'], epoch)
        writer.add_scalar('Accuracy/train', train_acc, epoch)
        writer.add_scalar('Accuracy/val', val_metrics['accuracy'], epoch)
        writer.add_scalar('AUC/val', val_metrics['auc'], epoch)
        writer.add_scalar('F1/val', val_metrics['f1'], epoch)
        writer.add_scalar('Precision/val', val_metrics['precision'], epoch)
        writer.add_scalar('Recall/val', val_metrics['recall'], epoch)
        writer.add_scalar('AP/val', val_metrics['ap'], epoch)
        writer.add_scalar('LR', optimizer.param_groups[0]['lr'], epoch)
        
        # Print epoch summary
        print(f"\nEpoch {epoch+1}/{cfg.epochs}:")
        print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.4f}")
        print(f"  AUC: {val_metrics['auc']:.4f} | F1: {val_metrics['f1']:.4f}")
        print(f"  Precision: {val_metrics['precision']:.4f} | Recall: {val_metrics['recall']:.4f}")
        print(f"  AP: {val_metrics['ap']:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        # Save best model
        if val_metrics['auc'] > best_metrics['auc']:
            best_metrics = val_metrics.copy()
            best_metrics['epoch'] = epoch + 1
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_metrics': val_metrics
            }, cfg.model_path)
            print("✅ New best model saved!")
            early_stop_counter = 0
        else:
            early_stop_counter += 1
        
        # Early stopping
        if cfg.early_stop and early_stop_counter >= cfg.patience:
            print(f"⏹ Early stopping triggered at epoch {epoch+1}")
            break

    # Save final model (last epoch's state)
    torch.save({
        'epoch': epoch + 1, # Use the last epoch reached
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_metrics': val_metrics # Last validation metrics
    }, cfg.last_model_path)
    
    # Save training history
    with open(os.path.join(cfg.exp_dir, "training_history.json"), 'w') as f:
        json.dump(history, f, indent=2)
    
    writer.close()
    return history, best_metrics

## 6. Evaluation and Visualization
This section provides functions to evaluate the trained model on validation and test sets, calculate various classification metrics, and generate insightful visualizations like confusion matrices, precision-recall curves, and training history plots.

In [28]:
def evaluate_model(model, loader, criterion=None, device=None):
    """Evaluate model performance and return metrics"""
    if device is None:
        device = next(model.parameters()).device
        
    model.eval()
    y_true, y_scores, y_pred = [], [], []
    total_loss = 0.0
    
    with torch.no_grad():
        for x, y in tqdm(loader, desc="Evaluating", leave=False):
            x, y = x.to(device), y.float().unsqueeze(1).to(device)
            
            with autocast(): # Use autocast for evaluation too if mixed precision was used in training
                preds = model(x)
                if criterion:
                    loss = criterion(preds, y)
                    total_loss += loss.item() * x.size(0)
            
            y_true.extend(y.cpu().numpy())
            y_scores.extend(torch.sigmoid(preds).cpu().numpy()) # Apply sigmoid to logits for scores
            y_pred.extend((preds > 0).int().cpu().numpy()) # Convert logits to binary predictions
    
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    y_pred = np.array(y_pred)
    
    metrics = {
        'loss': total_loss / len(loader.dataset) if criterion else 0.0,
        'accuracy': accuracy_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_scores),
        'f1': f1_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'ap': average_precision_score(y_true, y_scores)
    }
    
    return metrics, y_true, y_scores, y_pred

def generate_evaluation_report(model, loader, cfg, phase="val"):
    """Generate comprehensive evaluation report with visualizations"""
    metrics, y_true, y_scores, y_pred = evaluate_model(model, loader, device=next(model.parameters()).device)
    
    report = {
        'metrics': metrics,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'classification_report': classification_report(
            y_true, y_pred, target_names=cfg.class_names, output_dict=True
        )
    }
    
    # Save report
    report_path = os.path.join(cfg.exp_dir, f"{phase}_report.json")
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    
    # Plot confusion matrix
    plt.figure(figsize=(6, 6))
    sns.heatmap(report['confusion_matrix'], annot=True, fmt='d', cmap='Blues', 
                xticklabels=cfg.class_names, yticklabels=cfg.class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f"Confusion Matrix ({phase.capitalize()})")
    plt.savefig(os.path.join(cfg.exp_dir, f"{phase}_confusion_matrix.png"))
    plt.close()
    
    # Plot PR curve
    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    plt.figure(figsize=(6, 4))
    plt.plot(recall, precision, label=f"AP = {metrics['ap']:.2f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve ({phase.capitalize()})")
    plt.legend()
    plt.savefig(os.path.join(cfg.exp_dir, f"{phase}_pr_curve.png"))
    plt.close()

In [29]:
def plot_training_history(history, cfg):
    """Plot training metrics and save to experiment directory"""
    plt.figure(figsize=(18, 12))
    
    # Plot loss
    plt.subplot(2, 2, 1)
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Validation')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Plot accuracy
    plt.subplot(2, 2, 2)
    plt.plot(history['train_acc'], label='Train')
    plt.plot(history['val_acc'], label='Validation')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    # Plot AUC
    plt.subplot(2, 2, 3)
    plt.plot(history['val_auc'], label='Validation')
    plt.title('Validation AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    
    # Plot learning rate
    plt.subplot(2, 2, 4)
    plt.plot(history['lr'])
    plt.title('Learning Rate')
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.exp_dir, "training_metrics.png"))
    plt.close()

## 7. Configuration and Main Execution
This section defines the `Config` class to manage all hyperparameters and paths for the experiment. The main execution block orchestrates the entire process: initializing configuration, preparing data, building and training the model, plotting results, and evaluating on validation and test sets.

In [31]:
class Config:
    def __init__(self):
        # Data configuration
        self.data_dir = "dataset" # Path to your dataset (e.g., 'data/deepfake_faces')
        self.batch_size = 64  # Optimized for modern GPUs
        self.img_size = 256   # Better resolution for detection
        self.val_split = 0.15 # These splits are now indicative, actual split is by folders
        self.test_split = 0.15 # These splits are now indicative, actual split is by folders
        self.class_weights = True  # Handle class imbalance
        self.class_names = ["Real", "Fake"] # Default class names (will be updated by DataLoader)
        
        # Training configuration
        self.epochs = 30
        self.lr = 3e-4       # Optimized learning rate
        self.min_lr = 1e-6     # Minimum learning rate
        self.weight_decay = 1e-4  # Better regularization
        self.freeze_backbone = True
        self.freeze_epochs = 5  # Freeze initial layers
        self.dropout = 0.4     # Better regularization
        
        # Augmentation configuration
        self.augment = True
        self.color_jitter = 0.3
        self.random_erase_prob = 0.2
        
        # Optimization configuration
        self.optimizer = "AdamW"  # Best for this task
        self.scheduler = "CosineAnnealingWarmRestarts"  # Better learning rate adaptation
        self.patience = 5
        self.early_stop = False
        self.mixed_precision = True  # Faster training
        
        # Model checkpointing
        self.save_top_k = 3  # Save multiple checkpoints (not fully implemented in train_model, but good to have)
        
        # Experiment tracking
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.exp_name = f"mobilenetv3_{timestamp}" # Change model name as needed
        self.exp_dir = os.path.join("experiments", self.exp_name)
        os.makedirs(self.exp_dir, exist_ok=True)
        
        # Path configurations
        self.model_path = os.path.join(self.exp_dir, "best_model.pth")
        self.last_model_path = os.path.join(self.exp_dir, "last_model.pth")
        self.log_dir = os.path.join(self.exp_dir, "logs")
        
    def save(self):
        """Save configuration to experiment directory"""
        config_dict = {k:v for k,v in vars(self).items() if not k.startswith('__') and k != 'class_names'}
        with open(os.path.join(self.exp_dir, "config.json"), 'w') as f:
            json.dump(config_dict, f, indent=2)
            
    def __str__(self):
        return json.dumps(vars(self), indent=2)

if __name__ == "__main__":
    # Initialize configuration
    cfg = Config()
    cfg.save()
    print(f"\n⚙️ Configuration:\n{cfg}")
    
    # Prepare data
    print("\n📂 Loading and preparing data...")
    train_loader, val_loader, test_loader = prepare_dataloaders(cfg)
    
    # Build model (Choose one of the models to train)
    print("\n🧠 Building model...")
    # Example: MobileNetV3
    # model = build_mobilenetv3(freeze=cfg.freeze_backbone, dropout=cfg.dropout)
    # Example: ResNet50
    # model = build_resnet50(freeze=cfg.freeze_backbone, dropout=cfg.dropout)
    # Example: EfficientNetB0
    # model = build_efficientnetb0(freeze=cfg.freeze_backbone, dropout=cfg.dropout)
    # Example: DenseNet121
    # model = build_densenet121(freeze=cfg.freeze_backbone, dropout=cfg.dropout)
    
    print(f"Model architecture:\n{model}")
    # Optional: Print model summary
    try:
        # summary(model, input_size=(3, cfg.img_size, cfg.img_size), device=device.type) # Commented out to avoid 'list' object has no attribute 'size' error
        pass # Placeholder for commented out summary call
    except Exception as e:
        print(f"Could not print model summary: {e}")
    
    # Train model
    print("\n🏋️ Starting training...")
    history, best_metrics = train_model(model, train_loader, val_loader, cfg)
    
    # Plot training history
    plot_training_history(history, cfg)
    
    # Evaluate best model on validation set
    print("\n🔍 Evaluating best model on validation set...")
    checkpoint = torch.load(cfg.model_path, weights_only=False) # Added weights_only=False
    model.load_state_dict(checkpoint['model_state_dict'])
    val_report = generate_evaluation_report(model, val_loader, cfg, "val")
    
    # Evaluate on test set
    print("\n🧪 Evaluating on test set...")
    test_report = generate_evaluation_report(model, test_loader, cfg, "test")
    
    print(f"\n🎉 Training complete! Best validation AUC: {best_metrics['auc']:.4f}")
    print(f"📁 Results saved in: {cfg.exp_dir}")


⚙️ Configuration:
{
  "data_dir": "dataset",
  "batch_size": 64,
  "img_size": 256,
  "val_split": 0.15,
  "test_split": 0.15,
  "class_weights": true,
  "class_names": [
    "Real",
    "Fake"
  ],
  "epochs": 30,
  "lr": 0.0003,
  "min_lr": 1e-06,
  "weight_decay": 0.0001,
  "freeze_backbone": true,
  "freeze_epochs": 5,
  "dropout": 0.4,
  "augment": true,
  "color_jitter": 0.3,
  "random_erase_prob": 0.2,
  "optimizer": "AdamW",
  "scheduler": "CosineAnnealingWarmRestarts",
  "patience": 5,
  "early_stop": false,
  "mixed_precision": true,
  "save_top_k": 3,
  "exp_name": "mobilenetv3_20250707_041028",
  "exp_dir": "experiments\\mobilenetv3_20250707_041028",
  "model_path": "experiments\\mobilenetv3_20250707_041028\\best_model.pth",
  "last_model_path": "experiments\\mobilenetv3_20250707_041028\\last_model.pth",
  "log_dir": "experiments\\mobilenetv3_20250707_041028\\logs"
}

📂 Loading and preparing data...

🧠 Building model...


C:\Users\Shohan\AppData\Local\Temp\ipykernel_2944\474825013.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=cfg.mixed_precision)


Model architecture:
DenseNet121Binary(
  (base_model): DenseNet(
    (features): Sequential(
      (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu0): ReLU(inplace=True)
      (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (denseblock1): _DenseBlock(
        (denselayer1): _DenseLayer(
          (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu2): ReLU(inplace=True)
          (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (denselayer2): _DenseLayer(
          (norm1): BatchN

C:\Users\Shohan\AppData\Local\Temp\ipykernel_2944\474825013.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.mixed_precision):
C:\Users\Shohan\AppData\Local\Temp\ipykernel_2944\474825013.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.mixed_precision):
C:\Users\Shohan\AppData\Local\Temp\ipykernel_2944\474825013.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.mixed_precision):
C:\Users\Shohan\AppData\Local\Temp\ipykernel_2944\474825013.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.mixed_precision):
C:\Users\Shohan\AppData\Local\Temp\ipykernel_2944\474825


Epoch 1/30:
  Train Loss: 0.5819 | Acc: 0.5040
  Val Loss: 0.6867 | Acc: 0.5000
  AUC: 0.9052 | F1: 0.6667
  Precision: 0.5000 | Recall: 1.0000
  AP: 0.8548 | LR: 2.80e-04
✅ New best model saved!


...


Epoch 30/30:
  Train Loss: 0.5221 | Acc: 0.8183
  Val Loss: 0.5142 | Acc: 0.8941
  AUC: 0.9874 | F1: 0.9036
  Precision: 0.8297 | Recall: 0.9919
  AP: 0.9834 | LR: 2.80e-04

🔍 Evaluating best model on validation set...


C:\Users\Shohan\AppData\Local\Temp\ipykernel_2944\2999945105.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): # Use autocast for evaluation too if mixed precision was used in training
                                                                                                                       


🧪 Evaluating on test set...


C:\Users\Shohan\AppData\Local\Temp\ipykernel_2944\2999945105.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): # Use autocast for evaluation too if mixed precision was used in training
                                                                                                                       


🎉 Training complete! Best validation AUC: 0.9976
📁 Results saved in: experiments\mobilenetv3_20250707_041028
